# From logits to sampling: two experiments

**Review notebook.** We first isolate sampling using invented scores, then apply the same functions to the first-token scores of the model used in the original delivery article. This is an inference experiment: no model parameters are trained or updated. Run cells in order.

The second part needs PyTorch, Transformers, pandas, and the pinned Qwen model (about 1 GB download on a fresh machine; CPU float32 weights require roughly 2 GB of memory plus overhead). The first part only needs Python and pandas. See the existing [walkthrough setup](https://github.com/nabeelahmed/beyond-the-stack-notes/blob/main/what-is-an-llm/README.md). Model downloads may require network access and are cached outside the repository.

## From a prompt to a generated token

The model produces logits; the generation code uses them to select the next token. The two diagrams show these stages in order.

**1. The model processes the input and produces logits.**

```mermaid
flowchart TD
    START(["Start: new prompt"]) --> T["Tokenizer<br/>Prompt text → token IDs"]

    subgraph MODEL["Model forward pass"]
        direction TB
        E["Embedding layer"] --> B["All transformer blocks<br/>Self-attention + feed-forward networks"]
        B --> N["Final normalization"]
        N --> H["Output layer<br/>Vocabulary-sized logits"]
    end

    T -->|"First pass: prefill"| E
    H --> L["Last position's logits<br/>Scores for the next token"]
```

**2. The generation code uses those logits to select a token.**

```mermaid
flowchart TD
    L["Last position's logits"] --> D{"Decoding strategy"}

    subgraph SELECT["Token selection"]
        direction TB
        D -->|"Greedy"| G["Select highest-logit token"]
        D -->|"Sampling"| S["Apply temperature<br/>Optional top-k / top-p<br/>Sample from probabilities"]
    end

    G --> A["Append selected token ID"]
    S --> A
    A --> C{"Stop condition met?"}
    C -->|"Yes"| O["Convert generated token IDs<br/>to output text"]
    C -->|"No"| NEXT["Return to model for a decode forward pass<br/>with the selected token and KV cache"]
```

When the second diagram says **return to model**, the selected token goes back to the embedding layer in the first diagram. This subsequent **decode forward pass** uses the KV cache and produces logits for the following token. The initial **prefill** pass processes the prompt, produces logits for the first generated token, and builds the initial KV cache. Token selection then runs again after each decode forward pass until a stop condition is met.


### Why do people say “the LLM generates text”?

For the Qwen forward pass, the precise statement is:

> The model outputs logits. The generation code uses those logits to select the next token, and repeats the process to produce a response.

“LLM” is used at different levels of abstraction:

| What someone calls “the LLM” | What they usually mean | Output |
|---|---|---|
| The **model’s forward pass** | The neural network computation using its learned weights | Logits |
| The **model plus generation runtime** | The complete text-generation system | Generated token IDs |
| A **hosted LLM service** | The model, runtime, tokenizer and surrounding application | Text or a structured response |

These are common usages. In this notebook, we distinguish the model forward pass from the generation procedure so we can see where token selection happens.


### A single request: the model and generation runtime

The model boundary below is the neural network’s forward computation. The generation runtime calls the model, selects tokens, and manages the loop.

```mermaid
flowchart TD
    R(["User request"]) --> P["Prepare prompt<br/>Messages and conversation context"]
    P --> T["Tokenizer<br/>Prompt text → token IDs"]

    subgraph GEN["Generation runtime"]
        direction TB

        subgraph LLM["LLM model boundary"]
            M["Model forward pass<br/>Token IDs → logits"]
        end

        M --> S["Token selection<br/>Logits → next token ID"]
        S --> C{"Stop condition met?"}
        C -->|"No: next decode pass"| M
    end

    T -->|"First pass: prefill"| M
    C -->|"Yes"| D["Tokenizer<br/>Generated token IDs → text"]
    D --> O(["Response"])
```

The first model pass processes the prompt during **prefill**. If generation continues, each subsequent **decode forward pass** uses the selected token and typically a KV cache. Prompt preparation and response handling can vary by application.


## 1. A fixed, illustrative distribution

### Where does this happen?

For a decoder-only model such as the Qwen model later in this notebook, one next-token decision follows this sequence:

| Stage | Where | Receives | Produces |
|---|---|---|---|
| Tokenization | Before the model | Prompt text | Token IDs |
| Embedding and transformer blocks | Inside the model | Token IDs, converted to vectors | Contextual representations, one for each position |
| Final normalization and output projection | Inside the model, after the last transformer block | The last position's final representation | One score, or **logit**, per output token ID |
| Selection | Generation code, outside the transformer blocks | Logits and decoding settings | One selected next token ID |
| Repeat | Generation loop | Previous tokens plus the selected token | A new set of logits for the next decision |

**Sequence:** prompt → token IDs → transformer blocks → final representation → output projection → logits → selection → next token → repeat.

The blocks collectively produce the representation: it reflects the complete processing path, not attention alone. In a causal model the last position can use the preceding permitted context. Details vary by architecture; the output projection is a model layer after the transformer blocks, while the selection rules in this notebook are ordinary Python code outside the model.

### What are we doing in sections 1–4?

We isolate the selection stage with an **invented, complete vocabulary** of four pieces. We start with probabilities of 40%, 30%, 20%, and 10%, and work backwards using logarithms to create example logits. Softmax converts those logits back to our chosen probabilities. This backwards construction is for the demonstration: in actual generation the model gives us logits first.

For a single sampling decision, the calculation can be read in this order:

| Step | Input | Operation | Output |
|---|---|---|---|
| 1 | One logit per token | Optionally divide each logit by temperature | Adjusted logits |
| 2 | Adjusted logits | Softmax | A probability for each token; probabilities sum to 1 |
| 3 | Probabilities | Optionally keep candidates using top-k and/or top-p, then renormalize | A distribution over retained tokens |
| 4 | Final distribution | Draw according to its probabilities | One selected token |

The notebook compares the controls separately: section 2 changes temperature, while section 3 applies top-k and top-p separately to the original distribution. Section 4 draws repeatedly from each fixed distribution. Greedy decoding is another choice: it selects the largest logit directly and does not need softmax. If controls are combined, their order and exact behavior depend on the implementation; this notebook's helper uses temperature, then top-k, then top-p.

In [1]:
import math, random, platform, sys
from collections import Counter
import pandas as pd
from IPython.display import display

pieces = ['The', 'Order', 'A', 'Please']
example_probs = [0.4, 0.3, 0.2, 0.1]
example_logits = [math.log(p) for p in example_probs]

def softmax(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError('temperature must be positive')
    scaled = [x / temperature for x in logits]
    maximum = max(scaled)  # numerical stability; does not change the result
    weights = [math.exp(x - maximum) for x in scaled]
    total = sum(weights)
    return [x / total for x in weights]

def filter_distribution(probs, top_k=None, top_p=None):
    """Top-k then top-p; both operate on the current normalized distribution."""
    if top_k is not None and not 1 <= top_k <= len(probs):
        raise ValueError('top_k out of range')
    if top_p is not None and not 0 < top_p <= 1:
        raise ValueError('top_p must be in (0, 1]')
    ranked = sorted(range(len(probs)), key=lambda i: (-probs[i], i))
    if top_k is not None:
        ranked = ranked[:top_k]
    total = sum(probs[i] for i in ranked)
    if top_p is not None and top_p < 1:
        kept, cumulative = [], 0.0
        for i in ranked:
            kept.append(i)
            cumulative += probs[i] / total
            if cumulative >= top_p:
                break
        ranked = kept
    total = sum(probs[i] for i in ranked)
    allowed = set(ranked)
    return [p / total if i in allowed else 0.0 for i, p in enumerate(probs)]

def table(labels, probabilities):
    return pd.DataFrame({'token': labels, 'probability': probabilities})

base = softmax(example_logits)
assert all(abs(a-b) < 1e-12 for a,b in zip(base, example_probs))
display(pd.DataFrame({'token': pieces, 'logit': example_logits, 'probability': base}))

,token,logit,probability
0,The,-0.916291,0.4
1,Order,-1.203973,0.3
2,A,-1.609438,0.2
3,Please,-2.302585,0.1


## 2. Temperature changes relative probabilities

Temperature controls how strongly a random selection favors higher-scoring tokens. It adjusts the chances; it does not change the model's learned parameters or the logits the model originally produced.

- **Lower temperature** makes higher-scoring candidates more likely and results tend to be more predictable.
- **Temperature 1** gives the original distribution.
- **Higher temperature** makes the probabilities more even, giving lower-ranked candidates a greater chance.

The code divides each logit by a positive temperature and then applies softmax. For our invented example, the result is approximately:

| Token | T = 0.5 | T = 1 | T = 2 |
|---|---:|---:|---:|
| `The` | 53.3% | 40% | 32.5% |
| `Order` | 30.0% | 30% | 28.1% |
| `A` | 13.3% | 20% | 23.0% |
| `Please` | 3.3% | 10% | 16.3% |

Each column sums to approximately 100%. A lower temperature does not guarantee a correct answer. Greedy selection is a separate rule; some libraries use zero as shorthand for it.

In [2]:
temperatures = (0.5, 1.0, 2.0)
temp_table = pd.DataFrame({'token': pieces})
for t in temperatures:
    temp_table[f'T={t}'] = softmax(example_logits, t)
display(temp_table)
assert all(abs(sum(temp_table[f'T={t}']) - 1) < 1e-12 for t in temperatures)

    token     T=0.5  T=1.0     T=2.0
0     The  0.533333    0.4  0.325401
1   Order  0.300000    0.3  0.281805
2       A  0.133333    0.2  0.230093
3  Please  0.033333    0.1  0.162700


## 3. Select top candidates and renormalize

**Top-k keeps a fixed number of the most likely candidates.** With `top_k=2`, we keep `The` (40%) and `Order` (30%) and remove the other two. The kept probabilities add up to 70%, so we divide each by 70% to make the new distribution add up to 100%: `The` becomes about 57.1% and `Order` about 42.9%. This adjustment is called **renormalization**.

**Top-p keeps enough of the most likely candidates to reach a chosen probability threshold.** With `top_p=0.8`, we start with `The` (40%), add `Order` (cumulative 70%), then add `A` (cumulative 90%). We include `A` because it takes us past the 80% threshold. The three retained probabilities are divided by their 90% total, giving approximately 44.4%, 33.3%, and 22.2%.

| Token | Original | Top-k = 2 | Top-p = 0.8 |
|---|---:|---:|---:|
| `The` | 40% | 57.1% | 44.4% |
| `Order` | 30% | 42.9% | 33.3% |
| `A` | 20% | 0% | 22.2% |
| `Please` | 10% | 0% | 0% |

Top-k sets the **number of candidates**; top-p sets a **probability threshold**, so the number retained can change with the distribution. Temperature adjusts relative chances, while these filters remove candidates from consideration. The following cell compares the filters separately using the original distribution.

In [3]:
k_probs = filter_distribution(base, top_k=2)
p_probs = filter_distribution(base, top_p=0.8)
display(pd.DataFrame({'token': pieces, 'original': base, 'top_k=2': k_probs, 'top_p=0.8': p_probs}))
assert [i for i,p in enumerate(k_probs) if p] == [0,1]
assert [i for i,p in enumerate(p_probs) if p] == [0,1,2]
assert abs(sum(k_probs)-1) < 1e-12 and abs(sum(p_probs)-1) < 1e-12

    token  original   top_k=2  top_p=0.8
0     The       0.4  0.571429   0.444444
1   Order       0.3  0.428571   0.333333
2       A       0.2  0.000000   0.222222
3  Please       0.1  0.000000   0.000000


## 4. Decoding strategies: greedy or sampling

Greedy chooses the largest score every time. Sampling draws according to the current distribution. Repeated draws below use a fixed random seed and a **fixed prefix**: they are independent next-token choices, not complete responses. Counts fluctuate around expected frequencies; they do not measure response quality.

In [4]:
rng = random.Random(17)
draws = 10000
rows = []
for name, probs in [('base', base), ('temperature=0.5', softmax(example_logits, 0.5)), ('top-k=2', k_probs), ('top-p=0.8', p_probs)]:
    counts = Counter(rng.choices(pieces, weights=probs, k=draws))
    for token, probability in zip(pieces, probs):
        rows.append({'strategy': name, 'token': token, 'expected': probability, 'observed': counts[token]/draws})
print('Greedy:', pieces[max(range(len(example_logits)), key=example_logits.__getitem__)])
display(pd.DataFrame(rows))

Greedy: The
           strategy   token  expected  observed
0              base     The  0.400000    0.4011
1              base   Order  0.300000    0.3028
2              base       A  0.200000    0.1986
3              base  Please  0.100000    0.0975
4   temperature=0.5     The  0.533333    0.5368
5   temperature=0.5   Order  0.300000    0.2971
6   temperature=0.5       A  0.133333    0.1310
7   temperature=0.5  Please  0.033333    0.0351
8           top-k=2     The  0.571429    0.5642
9           top-k=2   Order  0.428571    0.4358
10          top-k=2       A  0.000000    0.0000
11          top-k=2  Please  0.000000    0.0000
12        top-p=0.8     The  0.444444    0.4436
13        top-p=0.8   Order  0.333333    0.3372
14        top-p=0.8       A  0.222222    0.2192
15        top-p=0.8  Please  0.000000    0.0000


## 5. A real model's first-token distribution

We now reuse the delivery prompt, chat template, model revision, CPU and float32 settings from the earlier article. This model produces scores for its entire configured output vocabulary. The five candidates displayed below are a **partial** list; their probabilities need not sum to one. The exact result is a measurement of this run, not a general claim about LLMs.

The notebook uses an already populated model cache by default (`LOCAL_ONLY = True`). Set it to `False` if the pinned model needs to be downloaded. No model outputs are sent to an external service.

In [5]:
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
LOCAL_ONLY = True  # Set False only if the pinned model needs to be downloaded.
torch.set_num_threads(2)
model_path = snapshot_download(MODEL, revision=REVISION, local_files_only=LOCAL_ONLY)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32,
    attn_implementation='eager', local_files_only=LOCAL_ONLY).to('cpu').eval()
model.requires_grad_(False)
prompt = ('Summarize this delivery update in one sentence to notify the customer:\n'
          '“Order 482 was scheduled for 18:00. The driver reports a van breakdown. '
          'The revised arrival time is 18:25.”')
messages = [{'role':'system', 'content':'You are a helpful assistant.'}, {'role':'user', 'content':prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                       return_dict=True, return_tensors='pt').to('cpu')
with torch.inference_mode():
    logits = model(**inputs).logits[0, -1].double()
probs = torch.softmax(logits, dim=-1)
values, ids = torch.topk(probs, 5)
print('Model:', MODEL, 'revision:', REVISION)
print('Python:', platform.python_version(), 'torch:', torch.__version__, 'transformers:', transformers.__version__)
print('Input tokens:', inputs['input_ids'].shape[1], '| output scores:', len(logits))
display(pd.DataFrame([{'token ID': i, 'piece': repr(tokenizer.decode([i])), 'logit': logits[i].item(), 'probability': v}
                      for i,v in zip(ids.tolist(),values.tolist())]))
print('Displayed probability mass:', values.sum().item())
assert abs(probs.sum().item()-1) < 1e-10

Model: Qwen/Qwen2.5-0.5B-Instruct revision: 7ae557604adf67be50417f59c2c2f167def9a775
Python: 3.9.6 torch: 2.8.0 transformers: 4.57.6
Input tokens: 68 | output scores: 151936
   token ID       piece      logit  probability
0       785       'The'  17.238058     0.298447
1      4431     'Order'  16.177197     0.103310
2        32         'A'  16.091616     0.094836
3      5501    'Please'  15.764205     0.068357
4     38121  'Delivery'  15.668959     0.062146
Displayed probability mass: 0.6270957069789407


## 6. Apply the same selection rules to the real distribution

To avoid losing the long tail, filters operate on **all** output scores. We show each method's retained candidate count and original probability mass, followed by its leading candidates after renormalization. Temperature is applied first; here the filters use `T=1`. A fixed seed makes this local sequence of draws repeatable under the same environment, but cross-platform reproducibility is not guaranteed.

In [6]:
real_logits = logits.tolist()
real_base = softmax(real_logits)
strategies = {'unfiltered': real_base,
              'temperature=0.5': softmax(real_logits, 0.5),
              'top-k=5': filter_distribution(real_base, top_k=5),
              'top-p=0.8': filter_distribution(real_base, top_p=0.8)}
rows=[]
for name, dist in strategies.items():
    kept = [i for i,p in enumerate(dist) if p > 0]
    for i in sorted(kept, key=lambda i: dist[i], reverse=True)[:5]:
        rows.append({'strategy':name, 'retained':len(kept), 'original mass retained':sum(real_base[j] for j in kept),
                     'token ID':i, 'piece':repr(tokenizer.decode([i])), 'new probability':dist[i]})
display(pd.DataFrame(rows))
rng = random.Random(17)
for name, dist in strategies.items():
    ids_drawn = rng.choices(range(len(dist)), weights=dist, k=8)
    print(name, [repr(tokenizer.decode([i])) for i in ids_drawn])
assert all(abs(sum(dist)-1) < 1e-9 for dist in strategies.values())
assert max(range(len(real_logits)), key=real_logits.__getitem__) == int(logits.argmax())

           strategy  retained  ...       piece  new probability
0        unfiltered    151936  ...       'The'         0.298447
1        unfiltered    151936  ...     'Order'         0.103310
2        unfiltered    151936  ...         'A'         0.094836
3        unfiltered    151936  ...    'Please'         0.068357
4        unfiltered    151936  ...  'Delivery'         0.062146
5   temperature=0.5    151936  ...       'The'         0.715421
6   temperature=0.5    151936  ...     'Order'         0.085725
7   temperature=0.5    151936  ...         'A'         0.072239
8   temperature=0.5    151936  ...    'Please'         0.037531
9   temperature=0.5    151936  ...  'Delivery'         0.031021
10          top-k=5         5  ...       'The'         0.475919
11          top-k=5         5  ...     'Order'         0.164743
12          top-k=5         5  ...         'A'         0.151231
13          top-k=5         5  ...    'Please'         0.109005
14          top-k=5         5  ...  'Del

## 7. Follow four generated tokens

Sections 5 and 6 inspected a single next-token distribution. What happens after we select a token? We append it to the context, run Qwen again, and inspect the new distribution. This experiment uses the same delivery prompt and model from section 5 and selects up to **four tokens using greedy decoding**.

At each step, the table shows the text generated so far, the five highest-scoring candidates and their probabilities, and the selected token. The probabilities are calculated over the full output vocabulary; the five displayed candidates are only a partial list. Token pieces may contain leading spaces, and a token is not necessarily a whole word.

The first selection reuses the logits computed during **prefill** in section 5. For the next selections, this simple walkthrough recomputes the full growing sequence with `use_cache=False`; it does not use a KV cache. A cached implementation can instead reuse prior attention state. This is inference: no model weights are updated. Generation stops early if the model selects its end-of-sequence token.


In [11]:
# Start from the original prompt and reuse the first-token logits from section 5.
token_ids = inputs['input_ids'].clone()
current_attention_mask = inputs['attention_mask'].clone()
next_token_logits = logits
input_prompt_length = token_ids.shape[1]
rows = []
eos_ids = model.generation_config.eos_token_id
if isinstance(eos_ids, int):
    eos_ids = [eos_ids]
eos_ids = set(eos_ids or [])

for step in range(1, 5):
    generated_before = tokenizer.decode(
        token_ids[0, input_prompt_length:], # all the generated tokens after the input sequence 
        skip_special_tokens=True,
    )
    
    selected_token_id = int(torch.argmax(next_token_logits))  # <-- Next token selection: Greedy approach, highest logit is selected
    selected_piece = tokenizer.decode([selected_token_id], skip_special_tokens=False)
    
    # generating also probabilities - for display only
    probabilities = torch.softmax(next_token_logits, dim=-1) 
    values, candidate_ids = torch.topk(probabilities, 5)

    for rank, (candidate_id, probability) in enumerate(zip(candidate_ids.tolist(), values.tolist()), 1):
        rows.append({
            'step': step,
            'generated before': repr(generated_before),
            'rank': rank,
            'candidate ID': candidate_id,
            'candidate piece': repr(tokenizer.decode([candidate_id], skip_special_tokens=False)),
            'probability': probability,
            'selected': candidate_id == selected_token_id,
        })

    next_id = torch.tensor([[selected_token_id]], dtype=token_ids.dtype, device=token_ids.device)
    token_ids = torch.cat([token_ids, next_id], dim=1)
    current_attention_mask = torch.cat([current_attention_mask, torch.ones_like(next_id)], dim=1)
    print(f"Step {step}: selected {selected_piece!r} (token ID {selected_token_id}); "
          f"generated so far: {tokenizer.decode(token_ids[0, input_prompt_length:], skip_special_tokens=True)!r}")
    if selected_token_id in eos_ids:
        print('Stopped: end-of-sequence token selected.')
        break
    if step < 4:
        with torch.inference_mode():
            model_output = model(
                input_ids=token_ids,
                attention_mask=current_attention_mask,
                use_cache=False,
            )
            all_logits = model_output.logits
            next_token_logits = all_logits[0, -1]  # <-- selecting the last token logits .. 0 is first sequence in batch, -1 last position in a sequence

display(pd.DataFrame(rows))


Step 1: selected 'The' (token ID 785); generated so far: 'The'
Step 2: selected ' delivery' (token ID 9691); generated so far: 'The delivery'
Step 3: selected ' was' (token ID 572); generated so far: 'The delivery was'
Step 4: selected ' scheduled' (token ID 13537); generated so far: 'The delivery was scheduled'


,step,generated before,rank,candidate ID,candidate piece,probability,selected
0,1,'',1,785,'The',0.298447,True
1,1,'',2,4431,'Order',0.103310,False
2,1,'',3,32,'A',0.094836,False
3,1,'',4,5501,'Please',0.068357,False
4,1,'',5,38121,'Delivery',0.062146,False
5,2,'The',1,9691,' delivery',0.243498,True
6,2,'The',2,6002,' customer',0.202497,False
7,2,'The',3,5579,' driver',0.181446,False
8,2,'The',4,1973,' order',0.134766,False
9,2,'The',5,220,' ',0.039882,False


## 8. Interpret the results

- Which candidates did temperature change most? How does restricting candidates differ from changing their relative probabilities?
- Does a token's high next-token probability establish that a full notification is factual or useful? Why not?
- Try another seed, temperature, or cutoff. Are the sample counts consistent with the probabilities as the draw count grows?
- A single first-token distribution cannot establish which strategy is better for a task. That would require a defined evaluation over representative prompts and full responses. Sampling order and exact cutoff behavior can vary between libraries; this notebook explicitly applies temperature, then top-k, then top-p.

Reference: [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751), which introduced nucleus sampling. [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).